## Introduction to the Network Flow Model

This notebook implements a network flow model using linear programming to optimize supply chain costs. The model aims to find the most cost-effective way to transport goods (or components) through a defined network of suppliers, assembly points, and distribution centers.

### Core Concepts:

*   **Vertices:** Represent locations or stages in the supply chain (e.g., 'source', 'China', 'Assembly_USA', 'Apple_HQ', 'US_Customers').
*   **Edges:** Represent the paths or routes between vertices. Each edge has a `weight` (cost of transport/production), an `upper` bound (maximum capacity), and an optional `lower` bound (minimum flow).

### Objective:

Minimize the total cost of moving goods through the supply chain while respecting all capacity constraints and demand requirements.

In [8]:
from scipy.optimize import linprog
import numpy as np
import math

vertices = ['source', 'China', 'Vietnam', 'Japan', 'India', 'Assembly_USA', 'Assembly_India', 'Apple_HQ']

edges = [
    # Supply of parts from countries
    {'from': 'source', 'to': 'China', 'weight': 91 + 4, 'upper': 100},
    {'from': 'source', 'to': 'South Korea', 'weight': 38 + 27, 'upper': 100},# base cost + tariff
    {'from': 'source', 'to': 'Vietnam', 'weight': 18 + 4, 'upper': 100},
    {'from': 'source', 'to': 'Japan', 'weight': 22, 'upper': 100},
    {'from': 'source', 'to': 'India', 'weight': 19, 'upper': 100},

    # Shipping to assembly
    {'from': 'China', 'to': 'Assembly_USA', 'weight': 6, 'upper': 100},
    {'from': 'Vietnam', 'to': 'Assembly_USA', 'weight': 7, 'upper': 80},
    {'from': 'Japan', 'to': 'Assembly_India', 'weight': 5, 'upper': 70},
    {'from': 'India', 'to': 'Assembly_India', 'weight': 4, 'upper': 90},

    # Final assembly shipping to HQ
    {'from': 'Assembly_USA', 'to': 'Apple_HQ', 'weight': 10, 'lower': 100},
    {'from': 'Assembly_India', 'to': 'Apple_HQ', 'weight': 8, 'lower': 100}
]

# Core functions from your original model
def sbv(index, size):
    return np.array([1.0 if i == index else 0.0 for i in range(size)])

def objective(edges):
    return sum([e["weight"] * sbv(edges.index(e), len(edges)) for e in edges])

def getIncoming(vertex, edges):
    return [e for e in edges if e["to"] == vertex]

def getOutgoing(vertex, edges):
    return [e for e in edges if e["from"] == vertex]

def isSource(vertex, edges):
    return getIncoming(vertex, edges) == []

def isSink(vertex, edges):
    return getOutgoing(vertex, edges) == []

def interiorVertices(vertices, edges):
    return [v for v in vertices if not (isSource(v, edges) or isSink(v, edges))]

def conservationLaw(vertex, edges):
    ii = sum([sbv(edges.index(e), len(edges)) for e in getIncoming(vertex, edges)])
    oo = sum([sbv(edges.index(e), len(edges)) for e in getOutgoing(vertex, edges)])
    return ii - oo

def conservationMatrix(vertices, edges):
    return np.array([conservationLaw(v, edges) for v in interiorVertices(vertices, edges)])

def lowerBound(edge):
    return edge.get('lower', -math.inf)

def upperBound(edge):
    return edge.get('upper', math.inf)

def ineqConstraints(edges):
    m = np.array([*[sbv(edges.index(e), len(edges)) for e in edges if upperBound(e) != math.inf],
                  *[-sbv(edges.index(e), len(edges)) for e in edges if lowerBound(e) != -math.inf]])
    b = np.array([*[upperBound(e) for e in edges if upperBound(e) != math.inf],
                 *[-lowerBound(e) for e in edges if lowerBound(e) != -math.inf]])
    return m, b

def runNetworkFlow(vertices, edges, maximize=False):
    obj = objective(edges)
    Aeq = conservationMatrix(vertices, edges)
    Aub, bub = ineqConstraints(edges)
    beq = np.zeros(len(interiorVertices(vertices, edges)))
    sgn = -1 if maximize else 1
    result = linprog(sgn * obj, A_eq=Aeq, b_eq=beq, A_ub=Aub, b_ub=bub)
    if result.success:
        optimal_value = sgn * result.fun
        return [f"Optimal Cost: ${optimal_value:.2f}"] + [
            (f"{e['from']} -> {e['to']}", float(result.x[edges.index(e)])) for e in edges
        ]
    else:
        return ["Optimization failed:", result.message]

# Run the model
output = runNetworkFlow(vertices, edges)
for line in output:
    print(line)

Optimal Cost: $8480.00
('source -> China', 20.0)
('source -> South Korea', 0.0)
('source -> Vietnam', 80.0)
('source -> Japan', 10.0)
('source -> India', 90.0)
('China -> Assembly_USA', 20.0)
('Vietnam -> Assembly_USA', 80.0)
('Japan -> Assembly_India', 10.0)
('India -> Assembly_India', 90.0)
('Assembly_USA -> Apple_HQ', 100.0)
('Assembly_India -> Apple_HQ', 100.0)


In [9]:
def display_supply_chain():
    vertices = ['source', 'Japan', 'Korea', 'Assembly_China', 'Assembly_India', 'Apple_HQ']
    edges = [
        {'from': 'source', 'to': 'Japan', 'weight': 25 + 3, 'upper': 100},  # cost + tariff
        {'from': 'source', 'to': 'Korea', 'weight': 38 + 2, 'upper': 100},

        {'from': 'Japan', 'to': 'Assembly_China', 'weight': 6, 'upper': 100},
        {'from': 'Korea', 'to': 'Assembly_India', 'weight': 5, 'upper': 100},

        {'from': 'Assembly_China', 'to': 'Apple_HQ', 'weight': 10, 'lower': 100},
        {'from': 'Assembly_India', 'to': 'Apple_HQ', 'weight': 8, 'lower': 100}
    ]
    return vertices, edges

def battery_supply_chain():
    vertices = ['source', 'Japan', 'Korea', 'Assembly_China', 'Assembly_India', 'Apple_HQ']
    edges = [
        {'from': 'source', 'to': 'Japan', 'weight': 24, 'upper': 100},  # cost + tariff
        {'from': 'source', 'to': 'Korea', 'weight': 24 + 2, 'upper': 100},

        {'from': 'Japan', 'to': 'Assembly_China', 'weight': 6, 'upper': 100},
        {'from': 'Korea', 'to': 'Assembly_India', 'weight': 5, 'upper': 100},

        {'from': 'Assembly_China', 'to': 'Apple_HQ', 'weight': 10, 'lower': 100},
        {'from': 'Assembly_India', 'to': 'Apple_HQ', 'weight': 8, 'lower': 100}
    ]
    return vertices, edges

def chipset_supply_chain():
    vertices = ['source', 'Japan', 'Korea', 'Assembly_China', 'Assembly_India', 'Apple_HQ']
    edges = [
        {'from': 'source', 'to': 'Japan', 'weight': 25 + 3, 'upper': 100},  # cost + tariff
        {'from': 'source', 'to': 'Korea', 'weight': 24 + 2, 'upper': 100},

        {'from': 'Japan', 'to': 'Assembly_China', 'weight': 6, 'upper': 100},
        {'from': 'Korea', 'to': 'Assembly_India', 'weight': 5, 'upper': 100},

        {'from': 'Assembly_China', 'to': 'Apple_HQ', 'weight': 10, 'lower': 100},
        {'from': 'Assembly_India', 'to': 'Apple_HQ', 'weight': 8, 'lower': 100}
    ]
    return vertices, edges

def camera_supply_chain():
    vertices = ['source', 'Japan', 'Korea', 'Assembly_China', 'Assembly_India', 'Apple_HQ']
    edges = [
        {'from': 'source', 'to': 'Japan', 'weight': 25 + 3, 'upper': 100},  # cost + tariff
        {'from': 'source', 'to': 'Korea', 'weight': 24 + 2, 'upper': 100},

        {'from': 'Japan', 'to': 'Assembly_China', 'weight': 6, 'upper': 100},
        {'from': 'Korea', 'to': 'Assembly_India', 'weight': 5, 'upper': 100},

        {'from': 'Assembly_China', 'to': 'Apple_HQ', 'weight': 10, 'lower': 100},
        {'from': 'Assembly_India', 'to': 'Apple_HQ', 'weight': 8, 'lower': 100}
    ]
    return vertices, edges

### Different Supply Chain Scenarios

The following code defines several supply chain scenarios for different components (Display, Battery, Chipset, Camera) and a final assembly stage. Each function (`display_supply_chain`, `battery_supply_chain`, etc.) returns a set of vertices and edges specific to that supply chain segment.

These functions allow for modular analysis of different parts of the overall supply chain, with varying costs, capacities, and origins/destinations.

**Common Elements:**

*   **`source`**: The starting point of components.
*   **Intermediate Locations**: Countries or assembly points (e.g., 'Japan', 'Korea', 'Assembly_China', 'Assembly_India').
*   **`Apple_HQ`**: The central distribution point.
*   **`weight`**: Cost associated with moving components along an edge (e.g., production cost, shipping cost, tariffs).
*   **`upper`**: Maximum flow capacity for an edge.
*   **`lower`**: Minimum required flow for an edge (often used to represent demand or minimum production).

In [10]:
def final_supply_chain():
    vertices = ['source', 'Assembly_USA', 'Assembly_India', 'Apple_HQ', 'US_Customers']
    edges = [
        {'from': 'source', 'to': 'Assembly_USA', 'weight': 30, 'upper': 400},  # combined cost from components
        {'from': 'source', 'to': 'Assembly_India', 'weight': 25, 'upper': 400},

        {'from': 'Assembly_USA', 'to': 'Apple_HQ', 'weight': 5, 'lower': 400},
        {'from': 'Assembly_India', 'to': 'Apple_HQ', 'weight': 4, 'lower': 400},

        {'from': 'Apple_HQ', 'to': 'US_Customers', 'weight': 2, 'lower': 400}
    ]
    return vertices, edges

In [11]:
for label, model in {
    "Display": display_supply_chain,
    "Battery": battery_supply_chain,
    "Chipset": chipset_supply_chain,
    "Camera": camera_supply_chain,
    "Final Assembly": final_supply_chain
}.items():
    v, e = model()
    print(f"--- {label} Supply Chain ---")
    result = runNetworkFlow(v, e)
    for line in result:
        print(line)

--- Display Supply Chain ---
Optimal Cost: $9700.00
('source -> Japan', 100.0)
('source -> Korea', 100.0)
('Japan -> Assembly_China', 100.0)
('Korea -> Assembly_India', 100.0)
('Assembly_China -> Apple_HQ', 100.0)
('Assembly_India -> Apple_HQ', 100.0)
--- Battery Supply Chain ---
Optimal Cost: $7900.00
('source -> Japan', 100.0)
('source -> Korea', 100.0)
('Japan -> Assembly_China', 100.0)
('Korea -> Assembly_India', 100.0)
('Assembly_China -> Apple_HQ', 100.0)
('Assembly_India -> Apple_HQ', 100.0)
--- Chipset Supply Chain ---
Optimal Cost: $8300.00
('source -> Japan', 100.0)
('source -> Korea', 100.0)
('Japan -> Assembly_China', 100.0)
('Korea -> Assembly_India', 100.0)
('Assembly_China -> Apple_HQ', 100.0)
('Assembly_India -> Apple_HQ', 100.0)
--- Camera Supply Chain ---
Optimal Cost: $8300.00
('source -> Japan', 100.0)
('source -> Korea', 100.0)
('Japan -> Assembly_China', 100.0)
('Korea -> Assembly_India', 100.0)
('Assembly_China -> Apple_HQ', 100.0)
('Assembly_India -> Apple_HQ', 

### Comprehensive Supply Chain Model

This section runs an example of a more comprehensive supply chain that incorporates multiple component suppliers, an assembly stage, and distribution. The `vertices` and `edges` for this model are defined directly within the cell.

This model demonstrates how various component suppliers (e.g., Processor_Taiwan, Display_Korea) feed into a central `Assembly` point, and then how the assembled product moves through `Distribution_US` to the `terminal` (representing the final customer).

**Key Features:**

*   **Component Sourcing**: Edges from 'source' to individual component suppliers, with `weight: 0` and `lower: 1` to ensure at least one unit of each component is considered.
*   **Assembly Cost**: Edges from component suppliers to 'Assembly' include the `weight` representing the cost of each component.
*   **Distribution**: Edges from 'Assembly' to 'Distribution_US' and then to 'terminal' represent the shipping and local delivery costs, with `upper` bounds indicating capacity.

In [12]:
 vertices = [
        'source',
        'Processor_Taiwan', 'Display_Korea', 'Battery_China',
        'Modem_China', 'Memory_US', 'Storage_Japan',
        'Camera_Supplier', 'Enclosure_Supplier', 'Misc_Suppliers',
        'Assembly',
        'Distribution_US',
        'terminal',
    ]

edges = [
        # From source to suppliers
        {'from': 'source', 'to': 'Processor_Taiwan', 'weight': 0, 'lower': 1},
        {'from': 'source', 'to': 'Display_Korea', 'weight': 0, 'lower': 1},
        {'from': 'source', 'to': 'Battery_China', 'weight': 0, 'lower': 1},
        {'from': 'source', 'to': 'Modem_China', 'weight': 0, 'lower': 1},
        {'from': 'source', 'to': 'Memory_US', 'weight': 0, 'lower': 1},
        {'from': 'source', 'to': 'Storage_Japan', 'weight': 0, 'lower': 1},
        {'from': 'source', 'to': 'Camera_Supplier', 'weight': 0, 'lower': 1},
        {'from': 'source', 'to': 'Enclosure_Supplier', 'weight': 0, 'lower': 1},
        {'from': 'source', 'to': 'Misc_Suppliers', 'weight': 0, 'lower': 1},

        # From suppliers to assembly (includes part cost as weight)
        {'from': 'Processor_Taiwan', 'to': 'Assembly', 'weight': 119.92, 'lower': 1},
        {'from': 'Display_Korea', 'to': 'Assembly', 'weight': 37.97, 'lower': 1},
        {'from': 'Battery_China', 'to': 'Assembly', 'weight': 47.46, 'lower': 1},
        {'from': 'Modem_China', 'to': 'Assembly', 'weight': 35.67, 'lower': 1},
        {'from': 'Memory_US', 'to': 'Assembly', 'weight': 21.80, 'lower': 1},
        {'from': 'Storage_Japan', 'to': 'Assembly', 'weight': 20.59, 'lower': 1},
        {'from': 'Camera_Supplier', 'to': 'Assembly', 'weight': 126.95, 'lower': 1},
        {'from': 'Enclosure_Supplier', 'to': 'Assembly', 'weight': 27.86, 'lower': 1},
        {'from': 'Misc_Suppliers', 'to': 'Assembly', 'weight': 240.06, 'lower': 1},

        # From assembly to U.S. distribution and customers
        {'from': 'Assembly', 'to': 'Distribution_US', 'weight': 0, 'upper': 100},  # shipping cost
        {'from': 'Distribution_US', 'to': 'terminal', 'weight':0, 'upper': 100},  # local delivery cost
    ]

runNetworkFlow(vertices, edges)


['Optimal Cost: $678.28',
 ('source -> Processor_Taiwan', 1.0),
 ('source -> Display_Korea', 1.0),
 ('source -> Battery_China', 1.0),
 ('source -> Modem_China', 1.0),
 ('source -> Memory_US', 1.0),
 ('source -> Storage_Japan', 1.0),
 ('source -> Camera_Supplier', 1.0),
 ('source -> Enclosure_Supplier', 1.0),
 ('source -> Misc_Suppliers', 1.0),
 ('Processor_Taiwan -> Assembly', 1.0),
 ('Display_Korea -> Assembly', 1.0),
 ('Battery_China -> Assembly', 1.0),
 ('Modem_China -> Assembly', 1.0),
 ('Memory_US -> Assembly', 1.0),
 ('Storage_Japan -> Assembly', 1.0),
 ('Camera_Supplier -> Assembly', 1.0),
 ('Enclosure_Supplier -> Assembly', 1.0),
 ('Misc_Suppliers -> Assembly', 1.0),
 ('Assembly -> Distribution_US', 9.0),
 ('Distribution_US -> terminal', 9.0)]